<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>AI SYSTEMS ENGINEERING 2</b>

## Unit 3: CrewAI — Tools, Memory, and Projects

**CV Raman Global University, Bhubaneswar**
*AI Center of Excellence*

</div>

---

### What You'll Learn

In this notebook, you will:

1. **Create custom tools** — use the `@tool` decorator to give agents abilities beyond text generation
2. **Equip agents with multiple tools** — let agents choose which tool to use for each subtask
3. **Enable memory** — allow agents to remember context across tasks
4. **Build a complete research crew** — combine agents, tools, memory, and pipelines into a working project
5. **Create reusable crew functions** — package crews as Python functions for repeated use

## 1. Environment Setup

In [ ]:
!pip install -q crewai

In [ ]:
import os
from getpass import getpass
from IPython.display import Markdown, display
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

print("Enter your API keys (press Enter to skip optional ones):\n")

google_key = getpass("Google Gemini API Key (recommended): ")
if google_key:
    os.environ['GEMINI_API_KEY'] = google_key
    print("Gemini key configured")

openai_key = getpass("OpenAI API Key (optional): ")
if openai_key:
    os.environ['OPENAI_API_KEY'] = openai_key
    print("OpenAI key configured")

groq_key = getpass("Groq API Key (optional): ")
if groq_key:
    os.environ['GROQ_API_KEY'] = groq_key
    print("Groq key configured")

In [ ]:
# CrewAI's LLM class uses LiteLLM under the hood
# llm = LLM(model="gemini/gemini-2.0-flash")

# To use a different provider, just change the model string:
llm = LLM(model="openai/gpt-4o-mini")
# llm = LLM(model="groq/llama-3.3-70b-versatile")

print(f"LLM configured: {llm.model}")

---

## 2. The Problem — Agents Need Abilities

An agent without tools is just a chatbot — it can only generate text from its training data. **Tools** let agents take actions: query databases, do math, call APIs, process files.

| Agent Alone | Agent with Tools |
|---|---|
| Can only generate text | Can call Python functions |
| Limited to training data | Can access real-time information |
| Guesses at calculations | Uses actual computation |
| No external actions | Can read files, call APIs, etc. |

CrewAI uses the `@tool` decorator to turn any Python function into an agent tool:

```
┌──────────────────────┐        ┌──────────────────────┐
│   Your Function      │        │   What the Agent Sees │
│                      │  ───>  │                      │
│  @tool               │        │  name: "calculator"   │
│  def calculator(     │        │  args: expression     │
│    expression: str   │        │  description: from    │
│  ) -> str:           │        │    the docstring      │
│    """Evaluates..."""│        │                      │
└──────────────────────┘        └──────────────────────┘
```

The LLM reads the tool's name and docstring, decides when to call it, and CrewAI handles execution.

---

## 3. Custom Tools with @tool

In [ ]:
# Create a calculator tool using the @tool decorator
@tool
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression and returns the result.
    
    Args:
        expression: A mathematical expression like '2 + 2' or '10 * 5'
    
    Returns:
        The result of the calculation
    """
    try:
        result = eval(expression)
        return f"The result of {expression} is {result}"
    except Exception as e:
        return f"Error calculating: {str(e)}"

print(f"Tool name: {calculator.name}")
print(f"Tool description: {calculator.description}")

In [ ]:
# Create an agent with the calculator tool
math_agent = Agent(
    role="Math Assistant",
    goal="Help with mathematical calculations",
    backstory="You are a helpful assistant that uses tools for accurate calculations.",
    tools=[calculator],
    llm=llm,
    verbose=True
)

math_task = Task(
    description="Calculate the following: What is 15 multiplied by 7, plus 23?",
    expected_output="The numerical answer to the calculation.",
    agent=math_agent
)

math_crew = Crew(
    agents=[math_agent],
    tasks=[math_task],
    verbose=True
)

result = math_crew.kickoff()
print(f"\nResult: {result}")

> **Key point:** The `@tool` decorator reads the function's **name**, **docstring**, and **type hints** to auto-generate a schema. Write clear docstrings — the LLM uses them to decide when and how to call your tool.

---

## 4. Multiple Tools on One Agent

Agents can have multiple tools and will **choose which to use** based on the task. This is similar to how we gave agents multiple `@function_tool`s in Unit 2.

In [ ]:
# Create a word counter tool
@tool
def word_counter(text: str) -> str:
    """Counts the number of words in a text.
    
    Args:
        text: The text to count words in
    
    Returns:
        The word count
    """
    count = len(text.split())
    return f"The text contains {count} words."

print(f"Tool created: {word_counter.name}")

In [ ]:
# Agent with multiple tools — it decides which to use
multi_tool_agent = Agent(
    role="Text Analyst",
    goal="Analyze and process text using available tools",
    backstory="You analyze text and can count words or perform calculations.",
    tools=[calculator, word_counter],  # Multiple tools!
    llm=llm,
    verbose=True
)

analysis_task = Task(
    description="""Analyze the text 'The Quick Brown Fox Jumps Over The Lazy Dog':
    1. Count how many words it has
    2. Calculate: number of words times 10
    Report both results.""",
    expected_output="Word count and the calculation result.",
    agent=multi_tool_agent
)

analysis_crew = Crew(
    agents=[multi_tool_agent],
    tasks=[analysis_task],
    verbose=True
)

result = analysis_crew.kickoff()

print("\nResult:")
display(Markdown(str(result)))

---

## 5. Memory — Agents That Remember

By default, each task runs independently. **Memory** (`memory=True` on the Crew) allows agents to remember information across tasks — useful when later tasks need context from earlier ones.

```
Without memory:   Task1 ──> Task2 (only sees Task1's output)
With memory:      Task1 ──> Task2 (sees Task1's output + remembers context)
```

In [ ]:
# Crew with memory + tools — tasks build on each other with richer context
assistant = Agent(
    role="Smart Assistant",
    goal="Help with calculations and text processing while remembering context",
    backstory="You are a capable assistant with access to calculation and text tools.",
    tools=[calculator, word_counter],
    llm=llm,
    verbose=True
)

task1 = Task(
    description="Calculate: 25 times 4.",
    expected_output="The numerical result.",
    agent=assistant
)

task2 = Task(
    description="Now count the words in: 'The answer from the previous calculation was very helpful and accurate.'",
    expected_output="The word count.",
    agent=assistant
)

memory_crew = Crew(
    agents=[assistant],
    tasks=[task1, task2],
    memory=True,  # Enable memory!
    verbose=True
)

print(f"Memory enabled: {memory_crew.memory}")

result = memory_crew.kickoff()
print(f"\nFinal Result: {result}")

> **When to use memory:** Enable it when your crew has multiple tasks that build on each other, especially in longer workflows where agents need to reference earlier findings.

---

## 6. Putting It All Together — Research Crew Project

Let's combine everything — agents, roles, tools, memory, and sequential processing — into a complete research crew:

```
User provides topic
        │
        ▼
┌───────────────────┐
│ Research Analyst   │  gathers information
│ (tools: database)  │
└────────┬──────────┘
         │ output flows
         ▼
┌───────────────────┐
│ Content Writer     │  writes the report
└────────┬──────────┘
         │
         ▼
   Final Article
```

In [ ]:
# A mock database tool for the researcher
@tool
def search_database(query: str) -> str:
    """Search an internal database for information on a topic.
    
    Args:
        query: The search query
    
    Returns:
        Relevant information from the database
    """
    # In production, this would query a real database or API
    database = {
        "renewable energy": "Solar capacity grew 30% in 2024. Wind energy now powers 10% of global electricity. Battery storage costs dropped 50% since 2020.",
        "electric vehicles": "Global EV sales reached 14 million in 2024. China leads with 60% market share. Average EV range now exceeds 300 miles.",
        "ai healthcare": "AI diagnostics accuracy reached 94% for certain cancers. Drug discovery time reduced by 40%. Telemedicine adoption grew 200% since 2020.",
    }
    for key, value in database.items():
        if key in query.lower():
            return f"Database results for '{query}': {value}"
    return f"No specific database entry for '{query}'. Use your general knowledge."

print(f"Tool created: {search_database.name}")

In [ ]:
# Define the topic
TOPIC = "The future of renewable energy"

# Research Analyst — has access to the database tool
researcher = Agent(
    role="Research Analyst",
    goal="Gather comprehensive and accurate information about the given topic",
    backstory="""You are an experienced research analyst with a keen eye for detail.
    You excel at finding relevant information and identifying key points.
    You always organize your findings in a clear, structured manner.""",
    tools=[search_database],
    llm=llm,
    verbose=True,
    allow_delegation=False
)

# Content Writer — no tools needed, just writing skill
writer = Agent(
    role="Content Writer",
    goal="Transform research findings into clear, engaging, and well-structured content",
    backstory="""You are a skilled content writer who excels at making complex topics
    accessible and interesting. You write in a clear, professional style.
    You structure content with proper headings and maintain reader engagement.""",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

print(f"Agents: {researcher.role}, {writer.role}")

In [ ]:
# Define tasks
research_task = Task(
    description=f"""Research the topic: {TOPIC}
    
    Your research should include:
    1. An overview of the current state
    2. Key trends and developments
    3. Main benefits or advantages
    4. Challenges or limitations
    5. Future outlook
    
    Use the search_database tool to find relevant data.""",
    expected_output="""A structured research document with:
    - Current state overview
    - Key trends (3-4 points)
    - Benefits (3-4 points)
    - Challenges (2-3 points)
    - Future outlook""",
    agent=researcher
)

writing_task = Task(
    description=f"""Based on the research provided, write a comprehensive but concise
    article about: {TOPIC}
    
    The article should:
    1. Have an engaging introduction
    2. Cover the main points from the research
    3. Be well-organized with clear sections
    4. End with a conclusion
    5. Be approximately 300-400 words""",
    expected_output="""A well-written article with:
    - Title
    - Introduction
    - Main body with sections
    - Conclusion
    - Approximately 300-400 words""",
    agent=writer
)

print("Tasks defined!")

In [ ]:
# Assemble and run the research crew
research_crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    process=Process.sequential,
    memory=True,
    verbose=True
)

print(f"Research crew assembled!")
print(f"Agents: {[a.role for a in research_crew.agents]}")
print(f"Process: Sequential | Memory: Enabled")
print("=" * 60)

result = research_crew.kickoff()

print("\n" + "=" * 60)
print("FINAL RESEARCH REPORT")
print("=" * 60 + "\n")
display(Markdown(str(result)))

### Making It Reusable

Package the crew as a function so you can research any topic with one call:

In [ ]:
def research_topic(topic: str, llm_model: str = "gemini/gemini-2.0-flash") -> str:
    """Run the research crew on a given topic."""
    llm = LLM(model=llm_model)
    
    researcher = Agent(
        role="Research Analyst",
        goal="Gather comprehensive information about the given topic",
        backstory="You are an experienced researcher who finds key information.",
        tools=[search_database],
        llm=llm,
        verbose=False
    )
    
    writer = Agent(
        role="Content Writer",
        goal="Write clear, engaging summaries",
        backstory="You excel at making complex topics accessible.",
        llm=llm,
        verbose=False
    )
    
    research_task = Task(
        description=f"Research the topic: {topic}. Find key facts, trends, benefits, and challenges.",
        expected_output="Structured research findings with key points.",
        agent=researcher
    )
    
    writing_task = Task(
        description=f"Write a 200-word summary about {topic} based on the research.",
        expected_output="A concise, well-written summary.",
        agent=writer
    )
    
    crew = Crew(
        agents=[researcher, writer],
        tasks=[research_task, writing_task],
        process=Process.sequential,
        memory=True,
        verbose=False
    )
    
    return crew.kickoff()

print("research_topic() function defined!")

In [ ]:
# Try it with a different topic
new_result = research_topic("Electric vehicles and their impact on transportation")

print("Result:")
display(Markdown(str(new_result)))

---

## 7. Key Takeaways

### Concept Map

```
Agent(role, goal, backstory, llm)
  │
  ├── tools=[...]           → agent can call Python functions
  │     └── @tool           → decorator auto-generates schema from docstring
  │
  └── assigned to ──> Task(description, expected_output, agent)
                        │
                        └── grouped into ──> Crew(agents, tasks, process, memory)
                                              │
                                              ├── memory=True  → agents remember context
                                              └── crew.kickoff() → Final Output
```

### Quick Reference

| Concept | Code | When to Use |
|---|---|---|
| **Create tool** | `@tool` decorator + docstring | Agent needs to call Python code |
| **Give tools** | `Agent(tools=[tool1, tool2])` | Agent needs abilities beyond text |
| **Enable memory** | `Crew(memory=True)` | Tasks build on each other |
| **Reusable crew** | Wrap in a Python function | Need to run the same workflow repeatedly |
| **allow_delegation** | `Agent(allow_delegation=False)` | Prevent agent from delegating to others |

---

## 8. Exercises

### Exercise 1: Custom Tool (Beginner)
Create a custom tool (e.g., `date_formatter`, `reverse_string`, or `random_number_generator`). Give it to an agent and test it with a relevant task.

### Exercise 2: Add an Editor Agent (Intermediate)
Extend the research crew from Section 6 by adding an **Editor** agent with a custom `grammar_check` tool. The editor should review and improve the writer's article. Your pipeline becomes: Researcher → Writer → Editor.

### Exercise 3: Multi-Model Research Crew (Intermediate)
Modify the `research_topic()` function to use **different LLM providers for each agent** — e.g., the researcher on `gemini/gemini-2.0-flash` (fast, cheap) and the writer on `openai/gpt-4o-mini` (strong writing). Compare the output quality to using a single model for both.

---

**Unit 3 Complete!** You've learned CrewAI fundamentals: agents, tasks, crews, execution modes, tools, memory, and built a complete research project.

---

**Course Information:**
- **Institution:** CV Raman Global University, Bhubaneswar
- **Program:** AI Center of Excellence
- **Course:** AI Systems Engineering 2
- **Developed by:** [Poorit Technologies](https://poorit.in) - *Transform Graduates into Industry-Ready Professionals*

---